In [11]:
import sys
from pathlib import Path

# ============================================================
# PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)

# ============================================================
# IMPORT RETRIEVER
# ============================================================

from rag.retriever import Retriever

retriever = Retriever(top_k=20)

Project root:
C:\Users\User\RAG SYS\Question_Generation
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 384
Loading FAISS index...
Loading metadata...

RETRIEVER INITIALIZED
Vectors: 2150
Metadata: 2150
Dimension: 384
Top-K: 20


In [12]:
query = "Explain object detection in computer vision."

results = retriever.retrieve(
    query,
    top_k=20
)

print("Number of results:", len(results))

print("\nFIRST RESULT:")
print(results[0])

print("\nFIRST RESULT KEYS:")
print(results[0].keys())

print("\nMETADATA:")
print(results[0].get("metadata"))

print("\nMETADATA KEYS:")
if isinstance(results[0].get("metadata"), dict):
    print(results[0]["metadata"].keys())

Number of results: 20

FIRST RESULT:
{'id': 'ai_ml_computer_vision_q021', 'score': 0.791130006313324, 'metadata': {'id': 'ai_ml_computer_vision_q021', 'metadata': {'id': 'ai_ml_computer_vision_q021', 'original_id': 'Q021', 'title': 'Object Detection', 'question': 'What is object detection?', 'type': 'technical', 'track': 'ai_ml', 'category': 'computer_vision', 'topic': 'object_detection', 'difficulty': 'medium', 'experience': ['junior'], 'skills': ['computer_vision'], 'language': 'en', 'duration': 90, 'source': 'question_bank', 'expected_concepts': ['Object localization', 'Bounding boxes', 'Class labels', 'Multiple objects'], 'source_file': 'ai_ml\\computer_vision.md'}}}

FIRST RESULT KEYS:
dict_keys(['id', 'score', 'metadata'])

METADATA:
{'id': 'ai_ml_computer_vision_q021', 'metadata': {'id': 'ai_ml_computer_vision_q021', 'original_id': 'Q021', 'title': 'Object Detection', 'question': 'What is object detection?', 'type': 'technical', 'track': 'ai_ml', 'category': 'computer_vision', '

In [13]:
query = "Explain object detection in computer vision."

results = retriever.retrieve(
    query,
    top_k=20
)

print("=" * 80)
print("QUERY:", query)
print("=" * 80)

for rank, result in enumerate(results, 1):

    metadata = result["metadata"]["metadata"]

    print(
        f"{rank}. "
        f"[{result['score']:.4f}] "
        f"{result['id']} - "
        f"{metadata['question']}"
    )

QUERY: Explain object detection in computer vision.
1. [0.7911] ai_ml_computer_vision_q021 - What is object detection?
2. [0.7547] ai_ml_computer_vision_q022 - What is the difference between image classification and object detection?
3. [0.7394] ai_ml_computer_vision_q028 - What is image segmentation?
4. [0.7385] ai_ml_computer_vision_q041 - What is face detection?
5. [0.7384] ai_ml_computer_vision_q023 - What is a bounding box in object detection?
6. [0.7322] ai_ml_computer_vision_q026 - How are precision and recall used to evaluate an object detector?
7. [0.7250] ai_ml_computer_vision_q001 - What is Computer Vision and what problems does it solve?
8. [0.7234] ai_ml_computer_vision_q042 - What is the difference between face detection and face recognition?
9. [0.7225] ai_ml_computer_vision_q002 - What are common applications of Computer Vision?
10. [0.7199] ai_ml_computer_vision_q030 - What is instance segmentation?
11. [0.7180] ai_ml_computer_vision_q038 - What is YOLO and why is it w

In [14]:
import sys
import pickle
import json
from pathlib import Path
from collections import defaultdict

# ============================================================
# PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ============================================================
# LOAD METADATA
# ============================================================

METADATA_FILE = (
    PROJECT_ROOT
    / "vector_store"
    / "metadata.pkl"
)

# If metadata.pkl is actually inside vector_store/
# use the path above.

with open(METADATA_FILE, "rb") as f:
    metadata = pickle.load(f)

print("Total records:", len(metadata))


# ============================================================
# GROUP QUESTIONS BY SOURCE FILE
# ============================================================

groups = defaultdict(list)

for item in metadata:

    # Your metadata structure:
    # item["id"]
    # item["metadata"]

    question_data = item["metadata"]

    question_id = question_data["id"]
    question = question_data["question"]
    source_file = question_data["source_file"]

    groups[source_file].append({
        "query": question,
        "expected_id": question_id,
        "source_file": source_file
    })


# ============================================================
# SELECT QUESTIONS FROM EVERY FILE
# ============================================================

QUESTIONS_PER_FILE = 5

evaluation_set = []

for source_file, questions in sorted(groups.items()):

    # Take the first 5 questions from each file
    selected = questions[:QUESTIONS_PER_FILE]

    evaluation_set.extend(selected)


# ============================================================
# SAVE
# ============================================================

TESTS_DIR = PROJECT_ROOT / "tests"
TESTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = TESTS_DIR / "retrieval_eval.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        evaluation_set,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# REPORT
# ============================================================

print("=" * 70)
print("RETRIEVAL EVALUATION DATASET")
print("=" * 70)

print("Question files:", len(groups))
print("Questions per file:", QUESTIONS_PER_FILE)
print("Total evaluation queries:", len(evaluation_set))

print("\nSaved to:")
print(OUTPUT_FILE)

print("\nExamples:")

for item in evaluation_set[:10]:

    print(
        f"\n[{item['source_file']}]"
        f"\nID: {item['expected_id']}"
        f"\nQuestion: {item['query']}"
    )

print("\n✓ Evaluation dataset created")

Total records: 2150
RETRIEVAL EVALUATION DATASET
Question files: 40
Questions per file: 5
Total evaluation queries: 200

Saved to:
C:\Users\User\RAG SYS\Question_Generation\tests\retrieval_eval.json

Examples:

[ai_ml\computer_vision.md]
ID: ai_ml_computer_vision_q001
Question: What is Computer Vision and what problems does it solve?

[ai_ml\computer_vision.md]
ID: ai_ml_computer_vision_q002
Question: What are common applications of Computer Vision?

[ai_ml\computer_vision.md]
ID: ai_ml_computer_vision_q003
Question: How is a digital image represented inside a computer?

[ai_ml\computer_vision.md]
ID: ai_ml_computer_vision_q004
Question: What is an RGB image?

[ai_ml\computer_vision.md]
ID: ai_ml_computer_vision_q005
Question: What is a grayscale image and how does it differ from an RGB image?

[ai_ml\deep_learning.md]
ID: ai_ml_deep_learning_q001
Question: What is deep learning and how does it differ from traditional machine learning?

[ai_ml\deep_learning.md]
ID: ai_ml_deep_learning_

In [15]:
import json
import sys
from pathlib import Path

# ============================================================
# PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag.retriever import Retriever


# ============================================================
# LOAD EVALUATION DATASET
# ============================================================

EVAL_FILE = PROJECT_ROOT / "tests" / "retrieval_eval.json"

with open(EVAL_FILE, "r", encoding="utf-8") as f:
    TEST_QUERIES = json.load(f)

print("Evaluation queries:", len(TEST_QUERIES))


# ============================================================
# INITIALIZE RETRIEVER
# ============================================================

retriever = Retriever()

print("Retriever loaded.")


# ============================================================
# METRICS
# ============================================================

hits_1 = 0
hits_3 = 0
hits_5 = 0
hits_10 = 0

reciprocal_ranks = []


# ============================================================
# EVALUATION
# ============================================================

for i, test in enumerate(TEST_QUERIES, 1):

    query = test["query"]
    expected_id = test["expected_id"]

    results = retriever.retrieve(
        query,
        top_k=10
    )

    retrieved_ids = [
        result["id"]
        for result in results
    ]

    # Recall@1
    if expected_id in retrieved_ids[:1]:
        hits_1 += 1

    # Recall@3
    if expected_id in retrieved_ids[:3]:
        hits_3 += 1

    # Recall@5
    if expected_id in retrieved_ids[:5]:
        hits_5 += 1

    # Recall@10
    if expected_id in retrieved_ids[:10]:
        hits_10 += 1

    # MRR
    if expected_id in retrieved_ids:

        rank = retrieved_ids.index(expected_id) + 1

        reciprocal_ranks.append(1 / rank)

    else:

        reciprocal_ranks.append(0)

    # Progress
    if i % 20 == 0:
        print(f"Evaluated {i}/{len(TEST_QUERIES)}")


# ============================================================
# FINAL METRICS
# ============================================================

total = len(TEST_QUERIES)

recall_1 = hits_1 / total
recall_3 = hits_3 / total
recall_5 = hits_5 / total
recall_10 = hits_10 / total

mrr = sum(reciprocal_ranks) / total


# ============================================================
# RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("BASELINE RETRIEVAL RESULTS")
print("=" * 70)

print(f"\nTest queries : {total}")

print(f"Recall@1     : {recall_1 * 100:.2f}%")
print(f"Recall@3     : {recall_3 * 100:.2f}%")
print(f"Recall@5     : {recall_5 * 100:.2f}%")
print(f"Recall@10    : {recall_10 * 100:.2f}%")
print(f"MRR          : {mrr:.4f}")

print("=" * 70)

Evaluation queries: 200
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 384
Loading FAISS index...
Loading metadata...

RETRIEVER INITIALIZED
Vectors: 2150
Metadata: 2150
Dimension: 384
Top-K: 20
Retriever loaded.
Evaluated 20/200
Evaluated 40/200
Evaluated 60/200
Evaluated 80/200
Evaluated 100/200
Evaluated 120/200
Evaluated 140/200
Evaluated 160/200
Evaluated 180/200
Evaluated 200/200


BASELINE RETRIEVAL RESULTS

Test queries : 200
Recall@1     : 95.50%
Recall@3     : 99.50%
Recall@5     : 100.00%
Recall@10    : 100.00%
MRR          : 0.9762


In [16]:
import sys
from pathlib import Path

# ============================================================
# PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag.retriever import Retriever


# ============================================================
# LOAD RETRIEVER
# ============================================================

retriever = Retriever()

print("=" * 80)
print("FULL RETRIEVAL EVALUATION")
print("=" * 80)

print(f"Total vectors: {len(retriever.metadata)}")

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 384
Loading FAISS index...
Loading metadata...

RETRIEVER INITIALIZED
Vectors: 2150
Metadata: 2150
Dimension: 384
Top-K: 20
FULL RETRIEVAL EVALUATION
Total vectors: 2150


In [17]:
# ============================================================
# EVALUATE ALL 2150 QUESTIONS
# ============================================================

total = len(retriever.metadata)

correct_at_1 = 0
correct_at_3 = 0
correct_at_5 = 0

reciprocal_ranks = []

failed = []


for i, item in enumerate(retriever.metadata):

    # Your metadata structure contains nested metadata
    metadata = item["metadata"]

    question_id = metadata["id"]
    question = metadata["question"]

    # --------------------------------------------------------
    # Retrieve using the ORIGINAL question
    # --------------------------------------------------------

    results = retriever.retrieve(
        question,
        top_k=5
    )

    retrieved_ids = [
        result["id"]
        for result in results
    ]

    # --------------------------------------------------------
    # Recall@1
    # --------------------------------------------------------

    if question_id in retrieved_ids[:1]:
        correct_at_1 += 1

    # --------------------------------------------------------
    # Recall@3
    # --------------------------------------------------------

    if question_id in retrieved_ids[:3]:
        correct_at_3 += 1

    # --------------------------------------------------------
    # Recall@5
    # --------------------------------------------------------

    if question_id in retrieved_ids[:5]:
        correct_at_5 += 1

    # --------------------------------------------------------
    # MRR
    # --------------------------------------------------------

    if question_id in retrieved_ids:

        rank = retrieved_ids.index(question_id) + 1
        reciprocal_ranks.append(1 / rank)

    else:

        reciprocal_ranks.append(0)

        failed.append({
            "id": question_id,
            "question": question,
            "retrieved": retrieved_ids
        })

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (i + 1) % 100 == 0:

        print(
            f"Processed {i + 1}/{total} "
            f"({((i + 1) / total) * 100:.1f}%)"
        )


# ============================================================
# FINAL METRICS
# ============================================================

recall_1 = correct_at_1 / total
recall_3 = correct_at_3 / total
recall_5 = correct_at_5 / total

mrr = sum(reciprocal_ranks) / total


print("\n")
print("=" * 80)
print("FULL RETRIEVAL RESULTS")
print("=" * 80)

print(f"Total questions: {total}")

print(f"Recall@1: {recall_1 * 100:.2f}%")
print(f"Recall@3: {recall_3 * 100:.2f}%")
print(f"Recall@5: {recall_5 * 100:.2f}%")
print(f"MRR:      {mrr:.4f}")

print("=" * 80)

Processed 100/2150 (4.7%)
Processed 200/2150 (9.3%)
Processed 300/2150 (14.0%)
Processed 400/2150 (18.6%)
Processed 500/2150 (23.3%)
Processed 600/2150 (27.9%)
Processed 700/2150 (32.6%)
Processed 800/2150 (37.2%)
Processed 900/2150 (41.9%)
Processed 1000/2150 (46.5%)
Processed 1100/2150 (51.2%)
Processed 1200/2150 (55.8%)
Processed 1300/2150 (60.5%)
Processed 1400/2150 (65.1%)
Processed 1500/2150 (69.8%)
Processed 1600/2150 (74.4%)
Processed 1700/2150 (79.1%)
Processed 1800/2150 (83.7%)
Processed 1900/2150 (88.4%)
Processed 2000/2150 (93.0%)
Processed 2100/2150 (97.7%)


FULL RETRIEVAL RESULTS
Total questions: 2150
Recall@1: 95.86%
Recall@3: 99.86%
Recall@5: 100.00%
MRR:      0.9786
